In [ ]:
Year=110
Period='AP'
discount=5/7

import geopandas as gpd
import os
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
import networkx as nx
from shapely.geometry import Point
from scipy.optimize import fsolve
import time

In [ ]:
#########################################
# 讀取 Shapefile 路網和節點資料
net= 'real' # 'sim' or 'real'
if net == 'sim':
    road_network = gpd.read_file('Net_V3.shp')
    #remove the link that Level exceeds 6 and the virtual link that Level is 12
    road_network = road_network[road_network['LEVEL'] <= 6]
    road_network_virtual = gpd.read_file('Net_V3.shp')
    road_network_virtual = road_network_virtual[road_network_virtual['LEVEL'] == 12]
    #centroids connectors
    road_network_centroid = gpd.read_file('Net_V3.shp')
    road_network_centroid = road_network_centroid[road_network_centroid['LEVEL'] == 11]
    # combine the link and the centroid
    road_network = pd.concat([road_network, road_network_centroid, road_network_virtual], ignore_index=True)
elif net == 'real':
    road_network = gpd.read_file('Y110Net - 複製.shp')
    road_network = road_network[road_network['LEVEL'] <= 6]
    road_network_centroid = gpd.read_file('Y110Net - 複製.shp')
    road_network_centroid = road_network_centroid[road_network_centroid['LEVEL'] == 11]
    road_network = pd.concat([road_network, road_network_centroid], ignore_index=True)

#for i in road_network['FFTIME'], if nan, set it to 0
road_network['FFTIME'] = road_network['FFTIME'].fillna(0)

zone_data = gpd.read_file('TZone601.shp')
zone_data = zone_data[zone_data['ID'] <=571]
zone_data = zone_data.sort_values(by=['TAZONENUM'])
zone_data.set_index('TAZONENUM', inplace=True)
#set the TAZ number as the index
TAZONENUM = list(set(zone_data.index.tolist()))

Nodes = []
for i in road_network['A'].values:
    if i not in Nodes:
        Nodes.append(i)
for i in road_network['B'].values:
    if i not in Nodes:
        Nodes.append(i)
Nodes.sort()

# 初始化有向圖
G = nx.DiGraph()

# 將邊加入圖中
for _, row in road_network.iterrows():
    G.add_edge(
        row['A'], row['B'],
        FFTIME=row['FFTIME'],
        CAPACITY=row['CAPACITY'],
        FFSPEED=row['FFSPEED'],
        LENGTH=row['LENGTH'],
        ALPHA=row['ALPHA'],
        BETA=row['BETA'],
        # toll=row['TOLL'],
        flow=0  # 初始化流量為0
    )

for u, v, data in G.edges(data=True):
    # 確保 FFSPEED 不為零
    if data['FFSPEED'] == 0:
        data['FFSPEED'] = 50  # 設定預設自由流速度（km/h）
    # 根據 FFSPEED 和長度計算 t0
    if data['FFTIME'] == 0:
        data['FFTIME'] = data['LENGTH'] / data['FFSPEED'] * 60  # 轉換為分鐘
    # 確保 t0 不為零
    if data['FFTIME'] == 0:
        data['FFTIME'] = 0.0001  # 設定一個很小的正數
################################################
def read_OD_matrix(year, period):
    file=f'OD_{year}_{period}.csv'
    All_np=np.loadtxt(file,delimiter=',')
    return All_np

OD=read_OD_matrix(Year, Period)
OD_df=pd.DataFrame(OD,index=TAZONENUM,columns=TAZONENUM)
zoneid_new=[25,26,27,28,29,30,31,32,33,34,35,36,38,39,45,46,47,50,51,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,112,37]

scenario = 'detail'
if scenario == 'detail':
    #North
    BeiDanSanShi=[171,172]+[i for i in range(174,184+1)]+[i for i in range(283,285+1)]+[376]+[i for i in range(442,458+1)]
    ShiTianShi=[i for i in range(1,16+1)]+[i for i in range(185,191+1)]+[173]

    #City center
    Daan=[i for i in range(40,72+1)]
    Zhongshan=[i for i in range(80,101+1)]
    Zhongzheng=[i for i in range(102,125+1)]+[38]
    Songshan=[i for i in range(192,213+1)]
    Xinyi=[i for i in range(214,243+1)]
    DatungShezi=[i for i in range(25,37+1)]+[39]+[i for i in range(17,24+1)]
    #South
    WenshanXinwuZhongYonghe=[i for i in range(152,170+1)]+[440]+[i for i in range(461,450+1)]+[i for i in range(461,490+1)]+[i for i in range(335,356+1)]+[i for i in range(368,375+1)]
    #East-West
    NeihuDazhi=[i for i in range(126,151+1)]+[i for i in range(73,79+1)]
    NangangXizhi=[i for i in range(244,259+1)]+[i for i in range(378,390+1)]
    BaWuLuChongxin=[i for i in range(275,282+1)]+[i for i in range(286,306+1)]+[i for i in range(357,366+1)]+[i for i in range(491,510+1)]+[i for i in range(529,537+1)]
    WanhuaBanTuShuSanYing=[i for i in range(260,274+1)]+[i for i in range(307,334+1)]+[i for i in range(406,430+1)]+[i for i in range(518,527+1)]+[i for i in range(538,552+1)]
    KeelungWanKingRuei=[431]+[i for i in range(511,517+1)]
    LinTaiTaoGui=[i for i in range(392,405+1)]+[i for i in range(432,439+1)]+[i for i in range(553,571+1)]
    country=[367,377,391,441,459,460,528]

    All=BeiDanSanShi+ShiTianShi+Daan+Zhongshan+Zhongzheng+Songshan+Xinyi+DatungShezi+WenshanXinwuZhongYonghe+NeihuDazhi+NangangXizhi+BaWuLuChongxin+WanhuaBanTuShuSanYing+KeelungWanKingRuei+LinTaiTaoGui+country

    #for all the zones in the network, i.e. zoneid, we reserve each zone, not merge them
    #except for the zones in the same area, we merge them as scenario 'Big' does
    BeiDanSanShi_dict={i:0 for i in BeiDanSanShi if i not in zoneid_new}
    ShiTianShi_dict={i:0 for i in ShiTianShi if i not in zoneid_new}

    Daan_dict={i:0 for i in Daan if i not in zoneid_new}
    Zhongshan_dict={i:0 for i in Zhongshan if i not in zoneid_new}
    Zhongzheng_dict={i:0 for i in Zhongzheng if i not in zoneid_new}
    Songshan_dict={i:0 for i in Songshan if i not in zoneid_new}
    Xinyi_dict={i:0 for i in Xinyi if i not in zoneid_new}
    DatungShezi_dict={i:0 for i in DatungShezi if i not in zoneid_new}
    NeihuDazhi_dict={i:0 for i in NeihuDazhi if i not in zoneid_new}

    WenshanXinwuZhongYonghe_dict={i:0 for i in WenshanXinwuZhongYonghe if i not in zoneid_new}

    NangangXizhi_dict={i:0 for i in NangangXizhi if i not in zoneid_new}
    BaWuLuChongxin_dict={i:0 for i in BaWuLuChongxin if i not in zoneid_new}
    WanhuaBanTuShuSanYing_dict={i:0 for i in WanhuaBanTuShuSanYing if i not in zoneid_new}

    network_dicts = {i:{i:0} for i in zoneid_new}
#reconstruct OD matrix by the classification of the TAZ
if scenario=='detail':
    corridor=['北淡','士林天母石牌','社子','大安','中正','松山','信義','文新雙和','內湖大直','南港汐止','重新','板土']
    for i in range(len(zoneid_new)):
        corridor.append(f'分區{zoneid_new[i]}')
    corridor_dict={'北淡':BeiDanSanShi_dict,'士林天母石牌':ShiTianShi_dict,'社子':DatungShezi_dict,'大安':Daan_dict,'中正':Zhongzheng_dict,'松山':Songshan_dict,'信義':Xinyi_dict,'文新雙和':WenshanXinwuZhongYonghe_dict,'內湖大直':NeihuDazhi_dict,'南港汐止':NangangXizhi_dict,'重新':BaWuLuChongxin_dict,'板土':WanhuaBanTuShuSanYing_dict}
    for i in range(len(zoneid_new)):
        corridor_dict[f'分區{zoneid_new[i]}']=network_dicts[zoneid_new[i]]
# Initialize the OD matrix as DataFrame
od = pd.DataFrame(np.zeros((len(corridor), len(corridor))), columns=corridor, index=corridor)
# Precompute the mapping of indices to corridors
index_to_corridor = {i: key for key in corridor_dict.keys() for i in corridor_dict[key]}
# Fill the OD matrix
obj=OD
for i in range(len(obj)):
    for j in range(len(obj[i])):
        key1 = index_to_corridor.get(i,0)
        key2 = index_to_corridor.get(j,0)
        if key1 and key2:
            od.loc[key1, key2] += obj[i, j]*discount
# The simplified TAZ (61 zones)
taz61={'北淡':7505,'士林天母石牌':6860,'社子':5860,'大安':5984,'中正':6076,'松山':6789,'信義':5325,'文新雙和':5983,'內湖大直':6812,'南港汐止':7037,'重新':5722,'板土':9424,
        '分區25':25, '分區26':26, '分區27':27, '分區28':28, '分區29':29, '分區30':30, '分區31':31, '分區32':32, '分區33':33, '分區34':34, '分區35':35, '分區36':36, '分區38':38, '分區39':39, '分區45':45, '分區46':46,
        '分區47':47, '分區50':50, '分區51':51, '分區80':80, '分區81':81, '分區82':82, '分區83':83, '分區84':84, '分區85':85, '分區86':86, '分區87':87, '分區88':88, '分區89':89, '分區90':90, '分區91':91, '分區92':92, '分區93':93, '分區94':94,
        '分區95':95, '分區96':96, '分區97':97, '分區98':98, '分區99':99, '分區100':100, '分區101':101, '分區102':102, '分區103':103, '分區104':104, '分區105':105, '分區106':106, '分區107':107, '分區112':112, '分區37':37}
taz61_inv = {v: k for k, v in taz61.items()}
######################################################
# 確認 OD 矩陣中的起訖對是否都對應到分區中的區心節點
od_assignments = []
# use od and taz61 to get the origin and destination node
for origin_zone in od.index:
    for dest_zone in od.columns:
        if origin_zone in taz61 and dest_zone in taz61:
            origin_node = taz61[origin_zone]
            dest_node = taz61[dest_zone]
            trips = od.at[origin_zone, dest_zone]
            # 建立起訖對的旅次指派清單
            od_assignments.append((origin_node, dest_node, trips))


# 確認輸出資料
# print("起訖對旅次分配:")
# for origin_node, dest_node, trips in od_assignments[110:120]:  # 僅顯示 10 筆
    # print(f"起點: {origin_node}, 終點: {dest_node}, 旅次量: {trips}")
print(f"總共 {len(od_assignments)} 筆旅次分配")

總共 3721 筆旅次分配


In [ ]:
start_time = time.time()

# 初始化道路參數
for u, v, data in G.edges(data=True):
    data['t0'] = data['LENGTH']/data['SPEED']  # 調整過的自由流時間（分鐘）
    data['capacity'] = data['CAPACITY']
    data['alpha'] = data['ALPHA']
    data['beta'] = data['BETA']
    data['length'] = data['LENGTH']
    data['flow'] = 0  # 初始流量
    data['time'] = data['t0']  # 初始旅行時間

# 定義 BPR 函數來更新旅行時間
def update_travel_times(G):
    for u, v, data in G.edges(data=True):
        volume = data['flow']
        capacity = data['capacity']
        alpha = data['alpha']
        beta = data['beta']
        t0 = data['t0'] if data['t0'] > 0 else 0.0001  # 確保 t0 不為零
        if capacity > 0 and t0 > 0:
            data['time'] = t0 * (1 + alpha * (volume / capacity) ** beta)
        else:
            data['time'] = max(t0 * 10, 0.0001)  # 避免時間為零

# 最大迭代次數
max_iterations = 20

for iteration in range(max_iterations):
    # 重置所有邊的流量
    for u, v, data in G.edges(data=True):
        data['flow'] = 0

    # 更新旅行時間
    update_travel_times(G)

    # 基於最短路徑分配流量
    for origin_node, dest_node, trips in od_assignments:
        if trips > 0:
            try:
                path = nx.bellman_ford_path(G, source=origin_node, target=dest_node, weight='time')
                # 將流量分配到路徑中的每一條邊
                for i in range(len(path) - 1):
                    u, v = path[i], path[i + 1]
                    G[u][v]['flow'] += trips
            except nx.NetworkXNoPath:
                print(f"無法在原點 {origin_node} 與目的地 {dest_node} 之間找到路徑")
                continue  # 無法在原點與目的地之間找到路徑

    print(f"第 {iteration + 1} 次迭代完成, 累計費時 {time.time() - start_time:.2f} 秒")
    # 如有需要，可在此處檢查收斂條件
    # 例如，檢查旅行時間的變化是否小於某個閾值

# 將結果寫回 shapefile
for u, v, data in G.edges(data=True):
    if data['capacity'] > 0:
        data['v_over_c'] = data['flow'] / data['capacity']
    else:
        data['v_over_c'] = 0  # 或設定為預設值
    data['assigned_speed'] = data['length'] / (data['time'] / 60)  # 計算指派後的速度（km/h）

第 1 次迭代完成, 累計費時 373.47 秒
第 2 次迭代完成, 累計費時 722.21 秒
第 3 次迭代完成, 累計費時 1085.49 秒
第 4 次迭代完成, 累計費時 1438.72 秒
第 5 次迭代完成, 累計費時 1787.30 秒
第 6 次迭代完成, 累計費時 3544.44 秒
第 7 次迭代完成, 累計費時 3873.43 秒
第 8 次迭代完成, 累計費時 4356.33 秒
第 9 次迭代完成, 累計費時 4824.93 秒
第 10 次迭代完成, 累計費時 5296.11 秒
第 11 次迭代完成, 累計費時 5794.34 秒
第 12 次迭代完成, 累計費時 6278.34 秒
第 13 次迭代完成, 累計費時 6784.59 秒
第 14 次迭代完成, 累計費時 7312.27 秒
第 15 次迭代完成, 累計費時 7751.59 秒
第 16 次迭代完成, 累計費時 8190.14 秒
第 17 次迭代完成, 累計費時 8625.48 秒
第 18 次迭代完成, 累計費時 9058.32 秒
第 19 次迭代完成, 累計費時 9553.25 秒
第 20 次迭代完成, 累計費時 12414.22 秒


第 1 次迭代完成, 累計費時 471.94 秒
第 2 次迭代完成, 累計費時 940.29 秒
第 3 次迭代完成, 累計費時 1423.02 秒
第 4 次迭代完成, 累計費時 1914.05 秒
第 5 次迭代完成, 累計費時 2422.59 秒
第 6 次迭代完成, 累計費時 2902.70 秒
第 7 次迭代完成, 累計費時 3432.40 秒
第 8 次迭代完成, 累計費時 3872.00 秒
第 9 次迭代完成, 累計費時 4314.42 秒
第 10 次迭代完成, 累計費時 4745.87 秒
第 11 次迭代完成, 累計費時 5179.32 秒
第 12 次迭代完成, 累計費時 5698.72 秒
第 13 次迭代完成, 累計費時 8512.64 秒
第 14 次迭代完成, 累計費時 8825.33 秒
第 15 次迭代完成, 累計費時 9158.37 秒
第 16 次迭代完成, 累計費時 9976.76 秒
第 17 次迭代完成, 累計費時 10304.96 秒
第 18 次迭代完成, 累計費時 10632.75 秒
第 19 次迭代完成, 累計費時 10973.70 秒
第 20 次迭代完成, 累計費時 11848.04 秒

第 1 次迭代完成, 累計費時 373.47 秒
第 2 次迭代完成, 累計費時 722.21 秒
第 3 次迭代完成, 累計費時 1085.49 秒
第 4 次迭代完成, 累計費時 1438.72 秒
第 5 次迭代完成, 累計費時 1787.30 秒
第 6 次迭代完成, 累計費時 3544.44 秒
第 7 次迭代完成, 累計費時 3873.43 秒
第 8 次迭代完成, 累計費時 4356.33 秒
第 9 次迭代完成, 累計費時 4824.93 秒
第 10 次迭代完成, 累計費時 5296.11 秒
第 11 次迭代完成, 累計費時 5794.34 秒
第 12 次迭代完成, 累計費時 6278.34 秒
第 13 次迭代完成, 累計費時 6784.59 秒
第 14 次迭代完成, 累計費時 7312.27 秒
第 15 次迭代完成, 累計費時 7751.59 秒
第 16 次迭代完成, 累計費時 8190.14 秒
第 17 次迭代完成, 累計費時 8625.48 秒
第 18 次迭代完成, 累計費時 9058.32 秒
第 19 次迭代完成, 累計費時 9553.25 秒
第 20 次迭代完成, 累計費時 12414.22 秒

第 1 次迭代完成, 累計費時 379.70 秒
第 2 次迭代完成, 累計費時 736.94 秒
第 3 次迭代完成, 累計費時 1146.56 秒
第 4 次迭代完成, 累計費時 1500.05 秒
第 5 次迭代完成, 累計費時 1947.18 秒
第 6 次迭代完成, 累計費時 2308.46 秒
第 7 次迭代完成, 累計費時 2816.60 秒
第 8 次迭代完成, 累計費時 3258.50 秒
第 9 次迭代完成, 累計費時 3592.51 秒
第 10 次迭代完成, 累計費時 3966.41 秒
第 11 次迭代完成, 累計費時 4324.98 秒
第 12 次迭代完成, 累計費時 4788.08 秒
第 13 次迭代完成, 累計費時 5294.68 秒
第 14 次迭代完成, 累計費時 5792.41 秒
第 15 次迭代完成, 累計費時 6293.58 秒
第 16 次迭代完成, 累計費時 6770.32 秒
第 17 次迭代完成, 累計費時 7162.85 秒
第 18 次迭代完成, 累計費時 7507.59 秒
第 19 次迭代完成, 累計費時 7852.21 秒
第 20 次迭代完成, 累計費時 8193.89 秒

第 1 次迭代完成, 累計費時 411.73 秒
第 2 次迭代完成, 累計費時 917.60 秒
第 3 次迭代完成, 累計費時 1374.16 秒

第 1 次迭代完成, 費時 297.51 秒
第 2 次迭代完成, 費時 566.31 秒
第 3 次迭代完成, 費時 835.25 秒
第 4 次迭代完成, 費時 1095.42 秒
第 5 次迭代完成, 費時 1365.36 秒
第 6 次迭代完成, 費時 1604.89 秒
第 7 次迭代完成, 費時 1850.26 秒
第 8 次迭代完成, 費時 2081.54 秒
第 9 次迭代完成, 費時 2313.46 秒
第 10 次迭代完成, 費時 2569.29 秒

第 1 次迭代完成, 費時 371.62 秒
第 2 次迭代完成, 費時 962.11 秒
第 3 次迭代完成, 費時 1567.64 秒
第 4 次迭代完成, 費時 1966.12 秒
第 5 次迭代完成, 費時 2329.35 秒
第 6 次迭代完成, 費時 2657.82 秒
第 7 次迭代完成, 費時 2983.89 秒
第 8 次迭代完成, 費時 3311.70 秒
第 9 次迭代完成, 費時 3881.47 秒
第 10 次迭代完成, 費時 4339.78 秒

In [ ]:
# 從 NetworkX 圖形中提取邊的屬性
edge_data = []

for u, v, data in G.edges(data=True):
    edge_data.append({
        'A': u,
        'B': v,
        'flow': data['flow'],
        'v_over_c': data['v_over_c'],
        'assigned_speed': data['assigned_speed'],
        'time': data['time']
    })

# 將邊的資料轉換為 DataFrame
edge_df = pd.DataFrame(edge_data)

# 假設 road_network 是您的原始路網 GeoDataFrame，包含 'A', 'B' 和 'geometry' 欄位
# 如果欄位名稱不同，請修改 'on' 參數中的欄位名稱

# 合併資料
merged_gdf = road_network.merge(edge_df, on=['A', 'B'], how='left')

# 設定 CRS（如果尚未設定）
merged_gdf.crs = road_network.crs

# 輸出檔案名稱
output_file = f'Complete_Assignment_Y{Year}_{Period}_{max_iterations}iter_PT50_demolition.shp'

# 將結果寫入 Shapefile
merged_gdf.to_file(output_file, driver='ESRI Shapefile')
print(f"結果已輸出至 {output_file}")

C:\Users\Allen\AppData\Local\Temp\ipykernel_19552\1592868841.py:30: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  merged_gdf.to_file(output_file, driver='ESRI Shapefile')
c:\Users\Allen\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'assigned_speed' to 'assigned_s'
  ogr_write(


結果已輸出至 Complete_Assignment_Y110_AP_20iter_PT50_demolition.shp
